# 📱 **Converting TensorFlow Model to TensorFlow Lite**
# **Deploy Your AI Models to Mobile Devices and Edge Computing**

---

## 🎓 **Learning Objectives**

By the end of this lesson, you will be able to:

✅ Understand **TensorFlow Lite (TFLite)** - Google's mobile/edge ML framework
✅ Convert **TensorFlow models** to optimized TFLite format
✅ Reduce **model size by 4×** through quantization
✅ Deploy models to **mobile devices** (Android, iOS)
✅ Run AI on **edge devices** (Raspberry Pi, microcontrollers)
✅ Optimize for **inference speed** and battery life
✅ Understand **FlatBuffer** format for efficient storage

---

## 📚 **Prerequisites**

Before starting this lesson, you should:

- ✅ Have built TensorFlow models (Lessons 5, 8, 9)
- ✅ Understand model saving/loading
- ✅ Know Sequential and Functional APIs
- ✅ Familiar with model deployment concepts

**Estimated Time:** 1.5 hours

---

## 🌟 **Why TensorFlow Lite?**

### **The Mobile AI Challenge:**

**Problem: Regular TensorFlow models are TOO BIG for mobile!**

```
Standard TensorFlow Model:
- Size: 500 MB (won't fit on phone!)
- Memory: 2 GB RAM needed
- Speed: 2 seconds per inference (too slow!)
- Battery: Drains in 1 hour
- Framework: 300 MB TensorFlow install

Result: ❌ Can't deploy to mobile devices
```

**Solution: TensorFlow Lite!**

```
TensorFlow Lite Model:
- Size: 10 MB (tiny!)
- Memory: 50 MB RAM needed
- Speed: 20ms per inference (60 FPS!)
- Battery: Runs all day
- Framework: 1 MB TFLite runtime

Result: ✅ Perfect for mobile & edge!
```

---

## 🎯 **What is TensorFlow Lite?**

### **Definition:**

**TensorFlow Lite** = Lightweight version of TensorFlow optimized for:
- **Mobile devices** (Android, iOS smartphones/tablets)
- **Edge devices** (Raspberry Pi, Jetson Nano)
- **Microcontrollers** (Arduino, ESP32)
- **IoT devices** (Smart cameras, sensors)

### **Key Features:**

| Feature | Regular TensorFlow | TensorFlow Lite |
|---------|-------------------|-----------------|
| **Model Size** | 100-500 MB | 5-20 MB (20-50× smaller!) |
| **Inference Speed** | 100-500ms | 10-50ms (10× faster!) |
| **Memory Usage** | 1-4 GB | 50-200 MB (20× less!) |
| **Platforms** | Desktop, Cloud | Mobile, Edge, MCU |
| **Runtime Size** | 300 MB | 1 MB (300× smaller!) |
| **Power Usage** | High | Low (battery-friendly) |

---

## 🏗️ **TensorFlow Lite Architecture**

### **The Conversion Pipeline:**

```
STEP 1: Train Model (Desktop/Cloud)
    TensorFlow/Keras
    ↓
    Your trained model.h5 (500 MB)

STEP 2: Convert to TFLite
    TFLiteConverter
    ↓
    Optimized model.tflite (20 MB)

STEP 3: Deploy to Device
    Android App / iOS App / Edge Device
    ↓
    Fast inference on device!
```

### **What Happens During Conversion:**

```
Original Model:
┌─────────────────────────┐
│ TensorFlow Graph        │
│ - Float32 weights       │
│ - Full operations       │
│ - All metadata          │
│ Size: 500 MB            │
└─────────────────────────┘
         ↓ CONVERT
┌─────────────────────────┐
│ TFLite FlatBuffer       │
│ - Quantized weights     │ ← 4× smaller
│ - Optimized ops         │ ← Faster
│ - Minimal metadata      │ ← Compact
│ Size: 20 MB             │
└─────────────────────────┘
```

---

## 📦 **FlatBuffer Format**

### **What is FlatBuffer?**

**FlatBuffer** = Google's efficient binary serialization format

**Why FlatBuffer vs Traditional Formats:**

| Format | Load Time | Memory | Use Case |
|--------|-----------|--------|----------|
| **JSON** | Slow (parse) | High | Human-readable |
| **Protocol Buffers** | Medium | Medium | Server communication |
| **FlatBuffer** | Instant | Minimal | **Mobile/Edge!** |

**Key Advantage:**
```
Traditional (JSON, Pickle):
1. Load file into memory
2. Parse entire file
3. Build object tree
4. Ready to use (slow!)

FlatBuffer:
1. Memory-map file
2. Ready to use! (instant!)
```

**No parsing needed = Instant loading = Perfect for mobile!**

---

## 🔧 **Three Conversion Methods**

### **Method 1: From Keras Model (Most Common)**

```python
# Load your trained model
model = tf.keras.models.load_model('my_model.h5')

# Convert
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save
with open('model.tflite', 'wb') as f:
    f.write(tflite_model)
```

**Use when:** You have a Keras model (.h5 or SavedModel format)

### **Method 2: From SavedModel (Recommended)**

```python
# Save model in SavedModel format
model.save('saved_model_dir')

# Convert
converter = tf.lite.TFLiteConverter.from_saved_model('saved_model_dir')
tflite_model = converter.convert()

# Save
with open('model.tflite', 'wb') as f:
    f.write(tflite_model)
```

**Use when:** Production deployment (SavedModel is TensorFlow's standard)

### **Method 3: From Concrete Functions (Advanced)**

```python
# For custom tf.function models
@tf.function(input_signature=[tf.TensorSpec(shape=[None], dtype=tf.float32)])
def my_function(x):
    return tf.square(x)

# Get concrete function
concrete_func = my_function.get_concrete_function()

# Convert
converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
tflite_model = converter.convert()
```

**Use when:** Custom low-level TensorFlow code, not Keras

---

## ⚡ **Optimization Techniques**

### **1. Default Conversion (No Optimization)**

```python
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Result: Same accuracy, smaller size
```

**Typical reduction:** 2-3× smaller
**Accuracy:** No loss
**Speed:** Slightly faster

### **2. Dynamic Range Quantization**

```python
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

# Result: Much smaller, same accuracy
```

**How it works:**
- Weights: float32 → int8 (4× smaller)
- Activations: Still float32
- No training data needed!

**Typical reduction:** 4× smaller
**Accuracy:** Minimal loss (<1%)
**Speed:** 2-3× faster

### **3. Full Integer Quantization**

```python
def representative_dataset():
    for _ in range(100):
        yield [np.random.rand(1, 224, 224, 3).astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8

tflite_model = converter.convert()

# Result: Smallest, fastest, runs on MCUs!
```

**How it works:**
- Weights: float32 → int8
- Activations: float32 → int8
- Requires representative dataset

**Typical reduction:** 4× smaller
**Accuracy:** Small loss (1-2%)
**Speed:** 3-4× faster
**Bonus:** Runs on microcontrollers!

---

## 📊 **Size Comparison Example**

**MobileNetV2 on ImageNet:**

| Version | Size | Accuracy | Speed (Pixel 4) |
|---------|------|----------|-----------------|
| **Original TF** | 14 MB | 72% | 180ms |
| **TFLite (default)** | 5 MB | 72% | 95ms |
| **Dynamic Quant** | 3.5 MB | 71.8% | 55ms |
| **Full Int8** | 3.5 MB | 71.3% | 38ms |

**Key Insight:** 4× smaller, 5× faster, 0.7% accuracy loss!

---

## 🚀 **Deployment Platforms**

### **1. Android (Java/Kotlin)**

```java
// Load model
Interpreter tflite = new Interpreter(loadModelFile());

// Prepare input
float[][] input = new float[1][224*224*3];

// Run inference
float[][] output = new float[1][1000];
tflite.run(input, output);

// Result: Predictions!
```

### **2. iOS (Swift)**

```swift
// Load model
let interpreter = try Interpreter(modelPath: modelPath)

// Allocate tensors
try interpreter.allocateTensors()

// Copy input
try interpreter.copy(Data(...), toInputAt: 0)

// Run
try interpreter.invoke()

// Get output
let output = try interpreter.output(at: 0)
```

### **3. Python (Edge Devices)**

```python
import tensorflow as tf

# Load TFLite model
interpreter = tf.lite.Interpreter(model_path="model.tflite")
interpreter.allocate_tensors()

# Get input/output details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Prepare input
input_data = np.array([[1.0, 2.0, 3.0]], dtype=np.float32)
interpreter.set_tensor(input_details[0]['index'], input_data)

# Run inference
interpreter.invoke()

# Get output
output_data = interpreter.get_tensor(output_details[0]['index'])
print(output_data)
```

### **4. Microcontrollers (C++)**

```cpp
// TensorFlow Lite Micro for Arduino, ESP32
#include "tensorflow/lite/micro/all_ops_resolver.h"
#include "tensorflow/lite/micro/micro_interpreter.h"

// Load model (embedded in flash)
const unsigned char model[] = { ... };

// Create interpreter
tflite::MicroInterpreter interpreter(model, resolver, tensor_arena, kTensorArenaSize);

// Run
interpreter.Invoke();
```

---

## 💡 **Real-World Use Cases**

### **1. 📱 Mobile Apps**

**Image Classification:**
- Google Photos (auto-tagging)
- Plant identification apps
- Food calorie estimation

**Object Detection:**
- Snapchat filters (face detection)
- Google Lens (object recognition)
- Parking apps (car detection)

**Natural Language:**
- Keyboard prediction
- Voice assistants (on-device)
- Language translation

### **2. 🏠 Edge Devices**

**Smart Home:**
- Security cameras (person detection)
- Smart doorbells (face recognition)
- Pet feeders (pet detection)

**Industrial:**
- Quality control (defect detection)
- Predictive maintenance
- Inventory management

### **3. 🤖 Robotics**

- Autonomous navigation
- Object manipulation
- Gesture recognition

### **4. 🚗 Automotive**

- Lane detection
- Pedestrian detection
- Driver drowsiness detection

---

Let's convert models for deployment! 🚀


## Steps to be followed:
1. Import the required libraries
2. Create and save the model
3. Convert the Keras model to a TensorFlow lite model
4. Convert concrete functions

### Step 1: Import the required libraries
- Use the popular open-source machine learning framework for building and training neural networks, and import the **tensorflow** library.


In [3]:
import tensorflow as tf
import numpy as np

### Step 2: Create and save the model
- Create a sequential model using the Keras API in TensorFlow.
- Compile the model with the specified optimizer and loss function.
- Fit the model to the training data for a specified number of epochs.
- Save the trained model in the specified directory using the TensorFlow **SavedModel** format.


In [4]:
# Define the model
model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(1,)),  # Use Input layer for the input shape
    tf.keras.layers.Dense(units=16, activation='relu'),
    tf.keras.layers.Dense(units=1)
])

# Compile the model
model.compile(optimizer='sgd', loss='mean_squared_error')

# Convert lists to NumPy arrays
x = np.array([-1, 0, 1])
y = np.array([-3, -1, 1])

# Train the model
model.fit(x=x, y=y, epochs=5)

# Save the model
tf.saved_model.save(model, 'saved_model_keras_dir')

Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 612ms/step - loss: 3.6110
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 3.4345
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 3.2596
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 3.0936
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 2.9360
INFO:tensorflow:Assets written to: saved_model_keras_dir/assets


INFO:tensorflow:Assets written to: saved_model_keras_dir/assets


__Observation:__

The code defines, trains, and saves a sequential Keras model in TensorFlow.

### Step 3: Convert the Keras model to a Tensorflow Lite model
- Save the TensorFlow Lite model to a file.



In [5]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open('model.tflite', 'wb') as f:
  f.write(tflite_model)

INFO:tensorflow:Assets written to: /tmp/tmp4_fd39fd/assets


INFO:tensorflow:Assets written to: /tmp/tmp4_fd39fd/assets


Saved artifact at '/tmp/tmp4_fd39fd'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 1), dtype=tf.float32, name='keras_tensor_4')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  140156862773760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140156862774464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140156863122944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140156863125408: TensorSpec(shape=(), dtype=tf.resource, name=None)


W0000 00:00:1730703995.175238     203 tf_tfl_flatbuffer_helpers.cc:392] Ignored output_format.
W0000 00:00:1730703995.175264     203 tf_tfl_flatbuffer_helpers.cc:395] Ignored drop_control_dependency.
2024-11-04 07:06:35.175580: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp4_fd39fd
2024-11-04 07:06:35.175875: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2024-11-04 07:06:35.175884: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmp4_fd39fd
2024-11-04 07:06:35.177940: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:388] MLIR V1 optimization pass is not enabled
2024-11-04 07:06:35.178379: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2024-11-04 07:06:35.196078: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmp4_fd39fd
2024-11-04 07:06:35.200692: I tensorflow/cc/saved_model/loader.cc

__Observation:__

The updated code converts a Keras model into a TensorFlow Lite model and saves it to a file named **model.tflite**.






### Step 4: Convert concrete functions
- Create a model using low-level `tf.` APIs.
- Define the custom TensorFlow module named **Squared** that squares input values.
- Create an instance of the Squared module.
- Get the concrete function from the module's **__call__** method.
- Convert the concrete function to a TensorFlow Lite model.



In [6]:
class Squared(tf.Module):
  @tf.function(input_signature=[tf.TensorSpec(shape=[None], dtype=tf.float32)])
  def __call__(self, x):
    return tf.square(x)

model = Squared()

concrete_func = model.__call__.get_concrete_function()

converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func],
                                                            model)
tflite_model = converter.convert()

INFO:tensorflow:Assets written to: /tmp/tmpntol75wp/assets


INFO:tensorflow:Assets written to: /tmp/tmpntol75wp/assets
W0000 00:00:1730704023.533468     203 tf_tfl_flatbuffer_helpers.cc:392] Ignored output_format.
W0000 00:00:1730704023.533489     203 tf_tfl_flatbuffer_helpers.cc:395] Ignored drop_control_dependency.
2024-11-04 07:07:03.533658: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpntol75wp
2024-11-04 07:07:03.533799: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2024-11-04 07:07:03.533808: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpntol75wp
2024-11-04 07:07:03.534611: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2024-11-04 07:07:03.538123: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpntol75wp
2024-11-04 07:07:03.540394: I tensorflow/cc/saved_model/loader.cc:462] SavedModel load for tags { serve }; Status: success: OK. Took 6738 m

In [7]:
result = model(tf.constant([2.0]))  # Example to square the value 2
print(result)  # Should print 4.0

tf.Tensor([4.], shape=(1,), dtype=float32)


**Observation**:

The provided code defines a TensorFlow module called **Squared** and converts it into a TensorFlow Lite model using the TensorFlow Lite converter.

**What This Demonstrates:**
- Low-level TensorFlow conversion (not Keras)
- Custom tf.function models
- Concrete function extraction
- TFLite supports custom operations!


---

## 🎓 **Key Takeaways & Summary**

### **What We Accomplished:**

✅ **Mastered TFLite conversion** - Deploy models to mobile/edge devices
✅ **Three conversion methods** - Keras, SavedModel, Concrete functions
✅ **Model optimization** - 4× smaller, 5× faster
✅ **FlatBuffer format** - Instant loading, minimal memory
✅ **Production deployment** - Android, iOS, Edge, MCU
✅ **Real-world applications** - Mobile apps, IoT, robotics

---

### **📊 Complete Conversion Workflow**

```python
# ========================================
# METHOD 1: From Keras Model (Simplest)
# ========================================

# Step 1: Train model
model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(784,)),
    tf.keras.layers.Dense(10, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
model.fit(x_train, y_train, epochs=5)

# Step 2: Convert to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Step 3: Save
with open('model.tflite', 'wb') as f:
    f.write(tflite_model)

# ========================================
# METHOD 2: From SavedModel (Recommended)
# ========================================

# Step 1: Save model
model.save('saved_model_dir')

# Step 2: Convert
converter = tf.lite.TFLiteConverter.from_saved_model('saved_model_dir')
tflite_model = converter.convert()

# Step 3: Save
with open('model.tflite', 'wb') as f:
    f.write(tflite_model)

# ========================================
# METHOD 3: With Quantization (Best)
# ========================================

converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Enable optimization
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Convert
tflite_model = converter.convert()

# Save
with open('model_quantized.tflite', 'wb') as f:
    f.write(tflite_model)

# Result: 4× smaller, 3× faster!
```

---

### **🔑 Key Concepts Mastered**

**1. TensorFlow Lite Benefits:**

```
Desktop TensorFlow → Mobile TFLite
─────────────────────────────────
Size:    500 MB   →   10 MB     (50× smaller!)
Speed:   500ms    →   50ms      (10× faster!)
Memory:  2 GB     →   100 MB    (20× less!)
Battery: 1 hour   →   All day   (10× better!)
```

**2. FlatBuffer Format:**

```
Traditional Formats (JSON, Pickle):
Load → Parse → Build object tree → Ready
(Slow!)

FlatBuffer:
Memory-map → Ready!
(Instant!)
```

**3. Quantization:**

```python
Float32 Weights:
[0.123456789, 0.987654321, -0.456789012, ...]
32 bits per number

Int8 Quantized:
[12, 99, -46, ...]
8 bits per number

Result: 4× smaller, minimal accuracy loss!
```

**4. Three Optimization Levels:**

| Optimization | Size Reduction | Speed Increase | Accuracy Loss |
|--------------|----------------|----------------|---------------|
| **None** | 2× | 1.5× | 0% |
| **Dynamic Range** | 4× | 3× | <1% |
| **Full Integer** | 4× | 4× | 1-2% |

---

### **💡 Practical Deployment Guide**

**Step 1: Choose Optimization Level**

```python
# For maximum accuracy (medical, safety-critical):
converter = tf.lite.TFLiteConverter.from_keras_model(model)
# No optimization

# For good balance (most apps):
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
# Dynamic range quantization

# For smallest/fastest (microcontrollers):
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
# Full integer quantization
```

**Step 2: Test Before Deployment**

```python
# Load TFLite model
interpreter = tf.lite.Interpreter(model_path="model.tflite")
interpreter.allocate_tensors()

# Get input/output details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Test with sample data
test_image = x_test[0:1]  # Get one test image
interpreter.set_tensor(input_details[0]['index'], test_image)
interpreter.invoke()
tflite_output = interpreter.get_tensor(output_details[0]['index'])

# Compare with original model
original_output = model.predict(test_image)

print(f"Original: {original_output}")
print(f"TFLite:   {tflite_output}")
print(f"Difference: {np.abs(original_output - tflite_output).max()}")

# If difference < 0.01: Good to deploy!
```

**Step 3: Measure Performance**

```python
import time

# Measure inference time
times = []
for _ in range(100):
    start = time.time()
    interpreter.invoke()
    times.append(time.time() - start)

print(f"Average inference time: {np.mean(times)*1000:.2f}ms")
print(f"FPS: {1/np.mean(times):.1f}")

# Check model size
import os
size_mb = os.path.getsize('model.tflite') / (1024 * 1024)
print(f"Model size: {size_mb:.2f} MB")
```

---

### **🚀 Platform-Specific Deployment**

**Android App (Kotlin):**

```kotlin
// 1. Add dependency to app/build.gradle:
implementation 'org.tensorflow:tensorflow-lite:2.13.0'

// 2. Put model.tflite in app/src/main/assets/

// 3. Load and use:
class ImageClassifier(context: Context) {
    private val interpreter: Interpreter

    init {
        val model = loadModelFile(context, "model.tflite")
        interpreter = Interpreter(model)
    }

    fun classify(bitmap: Bitmap): String {
        val input = preprocessImage(bitmap)
        val output = Array(1) { FloatArray(10) }

        interpreter.run(input, output)

        val classIdx = output[0].indices.maxByOrNull { output[0][it] } ?: 0
        return classes[classIdx]
    }
}
```

**iOS App (Swift):**

```swift
// 1. Add model.tflite to Xcode project

// 2. Load and use:
import TensorFlowLite

class ImageClassifier {
    private var interpreter: Interpreter

    init() throws {
        let modelPath = Bundle.main.path(forResource: "model", ofType: "tflite")!
        interpreter = try Interpreter(modelPath: modelPath)
        try interpreter.allocateTensors()
    }

    func classify(image: UIImage) throws -> String {
        let inputData = preprocess(image)
        try interpreter.copy(inputData, toInputAt: 0)
        try interpreter.invoke()

        let outputTensor = try interpreter.output(at: 0)
        let results = [Float](unsafeData: outputTensor.data) ?? []

        let maxIndex = results.indices.max(by: { results[$0] < results[$1] }) ?? 0
        return classes[maxIndex]
    }
}
```

**Raspberry Pi (Python):**

```python
# 1. Install TFLite runtime:
# pip install tflite-runtime

import tflite_runtime.interpreter as tflite
import numpy as np
from PIL import Image

# 2. Load model
interpreter = tflite.Interpreter(model_path="model.tflite")
interpreter.allocate_tensors()

# 3. Real-time camera inference
import cv2

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()

    # Preprocess
    image = cv2.resize(frame, (224, 224))
    image = np.expand_dims(image, axis=0).astype(np.float32) / 255.0

    # Inference
    interpreter.set_tensor(input_details[0]['index'], image)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])

    # Show result
    class_idx = np.argmax(output)
    cv2.putText(frame, classes[class_idx], (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.imshow('Classification', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
```

---

### **🎯 Challenge Exercises**

**1. Convert Your Own Model:**

```python
# Train a simple MNIST classifier
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.fit(x_train, y_train, epochs=5)

# Convert to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('mnist.tflite', 'wb') as f:
    f.write(tflite_model)

print(f"Original size: {model.count_params() * 4 / 1024 / 1024:.2f} MB")
print(f"TFLite size: {len(tflite_model) / 1024 / 1024:.2f} MB")
```

**2. Compare Optimization Levels:**

```python
import os

def convert_and_measure(model, name, optimization_level):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    if optimization_level == 'dynamic':
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
    elif optimization_level == 'full':
        def representative_dataset():
            for _ in range(100):
                yield [x_train[np.random.randint(0, len(x_train))][np.newaxis, ...].astype(np.float32)]

        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = representative_dataset

    tflite_model = converter.convert()
    filename = f'model_{name}.tflite'

    with open(filename, 'wb') as f:
        f.write(tflite_model)

    size = os.path.getsize(filename) / 1024
    print(f"{name}: {size:.2f} KB")

    return filename

# Test different levels
convert_and_measure(model, 'default', None)
convert_and_measure(model, 'dynamic', 'dynamic')
convert_and_measure(model, 'full_int', 'full')

# Compare accuracy
```

**3. Benchmark Inference Speed:**

```python
import time

def benchmark_model(model_path, num_runs=100):
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    # Prepare test input
    test_input = np.random.rand(*input_details[0]['shape']).astype(np.float32)

    # Warmup
    for _ in range(10):
        interpreter.set_tensor(input_details[0]['index'], test_input)
        interpreter.invoke()

    # Benchmark
    times = []
    for _ in range(num_runs):
        start = time.time()
        interpreter.set_tensor(input_details[0]['index'], test_input)
        interpreter.invoke()
        times.append(time.time() - start)

    avg_time = np.mean(times) * 1000  # ms
    fps = 1000 / avg_time

    print(f"Average time: {avg_time:.2f}ms")
    print(f"FPS: {fps:.1f}")
    print(f"Min: {min(times)*1000:.2f}ms, Max: {max(times)*1000:.2f}ms")

# Test
benchmark_model('model.tflite')
```

**4. Visualize Model Size Breakdown:**

```python
import tensorflow as tf

def analyze_tflite_model(model_path):
    # Load model
    with open(model_path, 'rb') as f:
        tflite_model = f.read()

    # Get model details
    interpreter = tf.lite.Interpreter(model_content=tflite_model)
    interpreter.allocate_tensors()

    # Count parameters
    total_params = 0
    for detail in interpreter.get_tensor_details():
        if 'shape' in detail:
            params = np.prod(detail['shape'])
            total_params += params
            print(f"{detail['name']}: {params} params")

    print(f"\nTotal parameters: {total_params:,}")
    print(f"Model size: {len(tflite_model) / 1024:.2f} KB")
    print(f"Bytes per parameter: {len(tflite_model) / total_params:.2f}")

analyze_tflite_model('model.tflite')
```

---

### **🔍 Debugging Tips**

**1. Model Too Large:**

```python
# Check model size
import os
size_mb = os.path.getsize('model.tflite') / (1024 * 1024)
print(f"Size: {size_mb:.2f} MB")

if size_mb > 10:  # Too large for mobile
    print("Solution: Apply quantization!")
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
```

**2. Accuracy Drop After Conversion:**

```python
# Test accuracy of TFLite model
def evaluate_tflite(model_path, x_test, y_test):
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    correct = 0
    for i in range(len(x_test)):
        interpreter.set_tensor(input_details[0]['index'], x_test[i:i+1])
        interpreter.invoke()
        output = interpreter.get_tensor(output_details[0]['index'])

        if np.argmax(output) == y_test[i]:
            correct += 1

    accuracy = correct / len(x_test)
    print(f"TFLite accuracy: {accuracy:.2%}")
    return accuracy

# If accuracy drops > 2%: Use lighter quantization or no quantization
```

**3. Unsupported Operations:**

```python
try:
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    tflite_model = converter.convert()
except Exception as e:
    print(f"Conversion error: {e}")
    print("Solution: Some ops not supported in TFLite")
    print("Try: converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]")
```

---

### **🌟 Production Best Practices**

**1. Version Your Models:**

```python
import datetime

# Add version to filename
version = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f'model_v{version}.tflite'

with open(filename, 'wb') as f:
    f.write(tflite_model)

# Keep metadata
metadata = {
    'version': version,
    'accuracy': test_accuracy,
    'size_mb': len(tflite_model) / (1024 * 1024),
    'quantization': 'dynamic_range'
}
```

**2. Validate Before Deployment:**

```python
def validate_tflite_model(original_model, tflite_path, x_test, threshold=0.01):
    # Test on same data
    original_pred = original_model.predict(x_test[:100])

    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    tflite_pred = []
    for img in x_test[:100]:
        interpreter.set_tensor(input_details[0]['index'], img[np.newaxis, ...])
        interpreter.invoke()
        tflite_pred.append(interpreter.get_tensor(output_details[0]['index'])[0])

    tflite_pred = np.array(tflite_pred)

    # Compare
    max_diff = np.abs(original_pred - tflite_pred).max()
    print(f"Maximum difference: {max_diff:.6f}")

    if max_diff < threshold:
        print("✅ Validation passed!")
        return True
    else:
        print("❌ Validation failed! Difference too large.")
        return False

# Use before deployment
validate_tflite_model(model, 'model.tflite', x_test)
```

**3. Monitor in Production:**

```python
# Log inference stats
class TFLitePredictor:
    def __init__(self, model_path):
        self.interpreter = tf.lite.Interpreter(model_path=model_path)
        self.interpreter.allocate_tensors()
        self.inference_count = 0
        self.total_time = 0

    def predict(self, input_data):
        start = time.time()

        # Run inference
        self.interpreter.set_tensor(input_details[0]['index'], input_data)
        self.interpreter.invoke()
        output = self.interpreter.get_tensor(output_details[0]['index'])

        # Log stats
        self.inference_count += 1
        self.total_time += time.time() - start

        return output

    def get_stats(self):
        avg_time = self.total_time / self.inference_count if self.inference_count > 0 else 0
        return {
            'count': self.inference_count,
            'avg_time_ms': avg_time * 1000,
            'fps': 1 / avg_time if avg_time > 0 else 0
        }
```

---

## 🏆 **Congratulations!**

You've mastered **TensorFlow Lite** - the key to deploying AI on mobile and edge devices!

**You now know:**
- ✅ TensorFlow Lite fundamentals (lightweight ML framework)
- ✅ Three conversion methods (Keras, SavedModel, Concrete)
- ✅ Quantization techniques (4× smaller, 3× faster)
- ✅ FlatBuffer format (instant loading)
- ✅ Platform deployment (Android, iOS, Edge, MCU)
- ✅ Production best practices (validation, monitoring)

**Skills Acquired:**
```
📱 Mobile Deployment:
- Android apps ✓
- iOS apps ✓
- Cross-platform ✓

⚡ Optimization:
- Dynamic range quantization ✓
- Full integer quantization ✓
- Size reduction (4×) ✓

🚀 Production:
- Model validation ✓
- Performance benchmarking ✓
- Version control ✓

🌍 Platforms:
- Smartphones ✓
- Tablets ✓
- Raspberry Pi ✓
- Microcontrollers ✓
```

**This knowledge applies to:**
- Mobile apps (billions of users!)
- IoT devices
- Edge computing
- Robotics
- Automotive
- Smart home
- Wearables
- Any resource-constrained deployment!

---

## 🎉 **The Complete Deep Learning Journey**

```
Lesson 3-4: Built networks from scratch
    ↓
Lesson 5: Mastered TensorFlow
    ↓
Lesson 8: Built CNNs for vision
    ↓
Lesson 9: Transfer Learning (95% accuracy)
    ↓
Lesson 10: Mobile Deployment! 📱

You're now PRODUCTION READY! 🚀
```

**Key Takeaway:**
> "TensorFlow Lite brings AI to billions of mobile devices. You can now deploy anywhere!"

**Next: Advanced topics or build your production app! 🌟**

---
